# Lab 8: Circuit-Level Noise and Repetitive Syndrome Extraction

**[EQE5006] Quantum Error Correction, Spring 2026**

Lab 7 stayed inside the **code-capacity** model: only `X_ERROR` on the data qubits, perfect ancillas, a single round of syndrome extraction. Real hardware does not give us those luxuries — *every* gate, reset, measurement, and idle period leaks errors, and noisy ancilla readout means a single round of detectors can no longer separate "the data qubits flipped" from "the ancilla measurement flipped".

In this lab we:

1. Introduce the **circuit-level noise model** in Stim (`DEPOLARIZE1/2`, measurement-flip, reset error).
2. Build a noisy 3-qubit repetition code with **repetitive syndrome extraction** using a `REPEAT` block, and extend the Lab 7 detector concept across rounds.
3. Decode the multi-round detector outcomes with PyMatching and sweep both the physical error rate $p$ and the number of rounds.
4. Use `stim.Circuit.generated` to build rotated surface codes under the same circuit-level noise model, and reproduce the canonical **threshold plot** as the code distance grows.

## Table of contents

0. [Recap and motivation](#sec-0)
1. [Circuit-level noise model in Stim](#sec-1)
2. [Noisy repetition code with REPEAT](#sec-2)
3. [Detectors for repetitive syndrome extraction](#sec-3)
4. [Simulation and decoding of the repetition code](#sec-4)
5. [Surface code threshold under circuit-level noise](#sec-5)


In [ ]:
# Install stim and PyMatching if not already available
!pip install stim pymatching -q


In [ ]:
import stim
import numpy as np
import matplotlib.pyplot as plt
import pymatching

<a id="sec-0"></a>

---

## Section 0: Recap and motivation

Lab 7 used the following minimal noise model on a 3-qubit bit-flip code:

- A single layer of `X_ERROR(p)` on the data qubits.
- A *perfect* ancilla-assisted syndrome extraction circuit.
- A single round of syndrome extraction before reading out the data.

Under those assumptions, two ancilla measurements completely characterize the error pattern (up to logical equivalence), and the lookup decoder fits in a 4-row table.

**What goes wrong on real hardware?**

- Two-qubit gates introduce correlated errors on both qubits.
- Single-qubit gates and idles still produce errors, just at a lower rate.
- Reset can leave the qubit in the wrong state.
- Measurement can return the wrong bit even if the qubit was in a well-defined eigenstate.

The first three issues mean that errors can be created **inside** the syndrome extraction circuit, not just on the data qubits before it. The last issue is the most striking: with a noisy ancilla readout, a single nonzero detector outcome could mean either "a data qubit flipped" or "the ancilla measurement flipped". A decoder cannot tell those apart from one round alone, and acting on the wrong interpretation introduces a *new* error.

The fix is to repeat the syndrome extraction **several times** and to compare consecutive rounds. A genuine data error keeps flipping the same ancilla outcome from then on, while a one-off measurement flip shows up as a *change* in the outcome that disappears in the next round.

This lab implements that idea step by step.


<a id="sec-1"></a>

---

## Section 1: Circuit-level noise model in Stim

Stim provides a family of noise instructions that we attach next to each physical operation. The conventions we will use throughout this lab — known as the **standard depolarizing model (SD6)** — are:

| Operation | Noise instruction | Where to place it | Note |
|:---|:---|:---|:---|
| Single-qubit Clifford (H, X, Z, …) | `DEPOLARIZE1(p)` | *after* the gate | |
| Two-qubit Clifford (CNOT, CZ) | `DEPOLARIZE2(p)` | *after* the gate | |
| Reset `R` or `RX` | `X_ERROR(p)` or `Z_ERROR(p)` | *after* the reset | |
| Measurement `M` or `MX` | `X_ERROR(p)` or `Z_ERROR(p)` | *before* the measurement | Can be inserted as `circuit.append('M', qubits, p)` |
| Idle qubit during a TICK | `DEPOLARIZE1(p)` | on every idling qubit | |

All noise rates are set to the **same** $p$ to keep things simple; real hardware models often use different rates for each channel.

- `DEPOLARIZE1(p)` applies one of $\{X, Y, Z\}$ with probability $p/3$ each (equivalently: any non-identity single-qubit Pauli with total probability $p$).
- `DEPOLARIZE2(p)` applies one of the $15$ non-identity two-qubit Paulis, each with probability $p/15$.
- `X_ERROR(p)` or `Z_ERROR(p)` flips the qubit with probability $p$ — the natural model for a Z-basis (X-basis) measurement or reset, since the only Pauli that matters is the one that anticommutes with $Z$ ($X$).

Let us see this in action on the familiar ancilla parity readout from Lab 7.

In [ ]:
# Perfect Bell state preparation

perfect_bell_state = stim.Circuit()
perfect_bell_state.append("R", [0, 1])
perfect_bell_state.append("TICK")
perfect_bell_state.append("H", [0])
perfect_bell_state.append("TICK")
perfect_bell_state.append("CNOT", [0, 1])
perfect_bell_state.append("TICK")

In [ ]:
# Clean check circuit with an ancillary qubit

clean_check = stim.Circuit()
clean_check.append("R", [2])
clean_check.append("TICK")
clean_check.append("CNOT", [0, 2])
clean_check.append("TICK")
clean_check.append("CNOT", [1, 2])
clean_check.append("TICK")
clean_check.append("M", [2])

In [ ]:
# A perfect ZZ parity readout on a perfect Bell state always returns +1.
clean_circuit = perfect_bell_state + clean_check
p_meas_clean = clean_circuit.compile_sampler().sample(shots=100_000)[:, 0].mean()
print(f"Clean ancilla flip rate: {p_meas_clean:.4f}")

In [ ]:
# Noisy check circuit with an ancillary qubit under SD6 noise model (excluding the idling noise)
p = 0.01
noisy_check = stim.Circuit()
noisy_check.append("R", [2])
noisy_check.append("X_ERROR", [2], p)
noisy_check.append("TICK")
noisy_check.append("CNOT", [0, 2])
noisy_check.append("DEPOLARIZE2", [0, 2], p)
noisy_check.append("TICK")
noisy_check.append("CNOT", [1, 2])
noisy_check.append("DEPOLARIZE2", [1, 2], p)
noisy_check.append("TICK")
noisy_check.append("M", [2], p)

noisy_circuit = perfect_bell_state + noisy_check
p_meas_noisy = noisy_circuit.compile_sampler().sample(shots=100_000)[:, 0].mean()
print(f"Noisy ancilla flip rate: {p_meas_noisy:.4f}")

Even though the underlying Bell state has $ZZ = +1$ deterministically, the noisy ancilla reports an outcome of $-1$ a few percent of the time. If we acted on that single bit as if it were a real error syndrome, we would inject a correction onto a perfectly fine codeword.

We can inspect *where* each noise instruction sits by drawing the circuit timeslice.

In [ ]:
noisy_circuit.diagram("timeline-svg")

### Exercise 1 - Add idling noise

The standard depolarizing model puts a `DEPOLARIZE1(p)` on every qubit that is **idling** during a time slice (i.e. between two consecutive `TICK`s, a qubit on which no gate is applied). The noisy circuit above is missing those idling terms.

Complete `noisy_check_with_idling_noise` below by inserting a `DEPOLARIZE1(p)` on every idle qubit just *before* each `TICK` boundary.

In [ ]:
# Exercise 1: insert idling noise on the qubits that did NOT participate
# in any gate during the preceding time slice.

p = 0.01
noisy_check_with_idling_noise = stim.Circuit()
noisy_check_with_idling_noise.append("R", [2])
noisy_check_with_idling_noise.append("X_ERROR", [2], p)
# TODO: Insert idling noise
noisy_check_with_idling_noise.append("TICK")

noisy_check_with_idling_noise.append("CNOT", [0, 2])
noisy_check_with_idling_noise.append("DEPOLARIZE2", [0, 2], p)
# TODO: Insert idling noise
noisy_check_with_idling_noise.append("TICK")

noisy_check_with_idling_noise.append("CNOT", [1, 2])
noisy_check_with_idling_noise.append("DEPOLARIZE2", [1, 2], p)
# TODO: Insert idling noise
noisy_check_with_idling_noise.append("TICK")

noisy_check_with_idling_noise.append("M", [2], p)

noisy_circuit_with_idling_noise = perfect_bell_state + noisy_check_with_idling_noise
p_meas_noisy_with_idle = (
    noisy_circuit_with_idling_noise.compile_sampler().sample(shots=100_000)[:, 0].mean()
)
print(f"Noisy ancilla flip rate: {p_meas_noisy_with_idle:.4f}")

<a id="sec-2"></a>

---

## Section 2: Noisy repetition code with `REPEAT`

We now build a *repetitive* syndrome extraction circuit for the 3-qubit bit-flip (repetition) code under circuit-level noise.

**Qubit layout** ($d = 3$):

- Data qubits: $q_0, q_1, q_2$ (logical $\bar{Z} = Z_0$).
- Ancilla qubits: $q_3$ measures $Z_0 Z_1$, $q_4$ measures $Z_1 Z_2$.

**Structure of one round of syndrome extraction:**

1. The ancillas have just been reset to $|0\rangle$ (with a reset-error `X_ERROR(p)`).
2. Two layers of CNOTs entangle each ancilla with the two data qubits it measures (with `DEPOLARIZE2(p)` on every CNOT).
3. The ancillas are measured (`X_ERROR(p)` *before* `M`), then reset again for the next round.

We will run this round `rounds` times using a `REPEAT` block. The first round is written outside the block so its detectors can compare the ancilla outcome to the known $|0\rangle$ eigenstate; later rounds compare to the previous round.

In [ ]:
# 3-qubit bit-flip code

p = 0.01
rounds = 3

data = [0, 1, 2]
anc = [3, 4]

c = stim.Circuit()

# ===== Initial reset of all qubits to |0> =====
c.append("R", data + anc)
c.append("X_ERROR", data + anc, p)
c.append("TICK")


# ----- One round of syndrome extraction
def one_round():
    syndrome_circuit = stim.Circuit()
    # Layer 1: CNOT 0 -> 3, 1 -> 4
    idling_qubits = set(data + anc)
    for i in range(2):
        syndrome_circuit.append("CNOT", [data[i], anc[i]])
        syndrome_circuit.append("DEPOLARIZE2", [data[i], anc[i]], p)
        idling_qubits.discard(data[i])
        idling_qubits.discard(anc[i])

    syndrome_circuit.append("DEPOLARIZE1", idling_qubits, p)  # idling noise

    syndrome_circuit.append("TICK")

    # Layer 2: CNOT 1 -> 3, 2 -> 4
    idling_qubits = set(data + anc)
    for i in range(2):
        syndrome_circuit.append("CNOT", [data[i + 1], anc[i]])
        syndrome_circuit.append("DEPOLARIZE2", [data[i + 1], anc[i]], p)
        idling_qubits.discard(data[i + 1])
        idling_qubits.discard(anc[i])

    syndrome_circuit.append("DEPOLARIZE1", idling_qubits, p)  # idling noise

    syndrome_circuit.append("TICK")

    # Measure the ancillas and then reset
    syndrome_circuit.append("MR", anc, p)  # equivalent to M(p) -> R
    syndrome_circuit.append("X_ERROR", anc, p)
    syndrome_circuit.append("DEPOLARIZE1", data, p)  # idling noise

    syndrome_circuit.append("TICK")
    return syndrome_circuit


c += one_round() * rounds

# ===== Final data Z measurement =====
c.append("M", data, p)

# Logical Z = Z on data[0].
c.append("OBSERVABLE_INCLUDE", [stim.target_rec(-3)], 0)

In [ ]:
print(c)

Notice the `REPEAT 3 { ... }` block in the printed listing — that is the second and third rounds of syndrome extraction collapsed into a single block. The first round sits outside the block so it can use simple single-record detectors.

In [ ]:
c.diagram("timeline-svg")

### Exercise 2 - $n$-qubit bit-flip code circuit

In [ ]:
# TODO: Generalize the above 3-qubit bit-flip code circuit to n-qubit bit-flip code

<a id="sec-3"></a>

---

## Section 3: Detectors for repetitive syndrome extraction

Lab 7 used `DETECTOR` to flag a *single* ancilla measurement whose outcome should be predictable from the prior state. With multi-round syndrome extraction we have to generalise: each detector now combines measurement records that, together, should be deterministic in the noiseless case.

Let $m_i^{(r)} \in \{0, 1\}$ be the outcome bit of ancilla $i$ in round $r$. We use three different detector definitions in this circuit:

**1. First round** — the ancilla was just reset, so $m_i^{(0)} = 0$ in the noiseless case:

$$D_i^{(0)} = m_i^{(0)}.$$

**2. Subsequent rounds** — the ancilla outcome should match the previous round if no data error happened in between:

$$D_i^{(r)} = m_i^{(r)} \oplus m_i^{(r-1)} \quad (r \geq 1).$$

This is the key change relative to Lab 7: a single data error makes the *first* round after the error fire, but every subsequent round matches again, while a measurement flip makes only one round fire on its own.

**3. Final detectors after data measurement** — we measure all data qubits in the Z basis, then reconstruct what the ancilla *would have* reported in a final, perfect round:

$$\tilde m_i = d_i \oplus d_{i+1},$$

and XOR this reconstructed value against the last real ancilla outcome:

$$D_i^{(\text{final})} = d_i \oplus d_{i+1} \oplus m_i^{(R-1)},$$

where $R$ is the total number of rounds.

In [ ]:
def make_rep_circuit(p, rounds, distance=3):
    """Noisy repetition code memory-Z circuit with repetitive SE."""
    n_data = distance
    n_anc = distance - 1
    data = list(range(n_data))
    anc = list(range(n_data, n_data + n_anc))

    c = stim.Circuit()

    # ===== Initial reset of all qubits to |0> =====
    c.append("R", data + anc)
    c.append("X_ERROR", data + anc, p)
    c.append("TICK")

    # ----- One round of syndrome extraction, written as a helper.
    def one_round(first):
        syndrome_circuit = stim.Circuit()
        # Layer 1: CNOT data[i] -> anc[i]
        idling_qubits = set(data + anc)
        for i in range(2):
            syndrome_circuit.append("CNOT", [data[i], anc[i]])
            syndrome_circuit.append("DEPOLARIZE2", [data[i], anc[i]], p)
            idling_qubits.discard(data[i])
            idling_qubits.discard(anc[i])

        syndrome_circuit.append("DEPOLARIZE1", idling_qubits, p)  # idling noise

        syndrome_circuit.append("TICK")

        # Layer 2: CNOT data[i+1] -> anc[i]
        idling_qubits = set(data + anc)
        for i in range(2):
            syndrome_circuit.append("CNOT", [data[i + 1], anc[i]])
            syndrome_circuit.append("DEPOLARIZE2", [data[i + 1], anc[i]], p)
            idling_qubits.discard(data[i + 1])
            idling_qubits.discard(anc[i])

        syndrome_circuit.append("DEPOLARIZE1", idling_qubits, p)  # idling noise

        syndrome_circuit.append("TICK")

        # Measure the ancillas and then reset
        syndrome_circuit.append("MR", anc, p)  # equivalent to M(p) -> R
        syndrome_circuit.append("X_ERROR", anc, p)
        syndrome_circuit.append("DEPOLARIZE1", data, p)  # idling noise

        # Add detectors
        for i_anc in range(n_anc):
            if first:
                syndrome_circuit.append("DETECTOR", [stim.target_rec(-n_anc + i_anc)])
            else:
                syndrome_circuit.append(
                    "DETECTOR",
                    [
                        stim.target_rec(-n_anc + i_anc),
                        stim.target_rec(-2 * n_anc + i_anc),
                    ],
                )

        syndrome_circuit.append("TICK")
        return syndrome_circuit

    # ===== First round (compare to known |0> ancillas) =====
    c += one_round(True)

    # ===== Remaining (rounds - 1) rounds inside a REPEAT block =====
    if rounds > 1:
        c += one_round(False) * (rounds - 1)

    # ===== Final data Z measurement =====
    c.append("M", data, p)

    # Final detectors
    for i in range(n_anc):
        c.append(
            "DETECTOR",
            [
                stim.target_rec(-n_data + i),  # data[i]
                stim.target_rec(-n_data + i + 1),  # data[i+1]
                stim.target_rec(-n_data - n_anc + i),  # last anc[i]
            ],
        )

    # Logical Z = Z on data[0].
    c.append("OBSERVABLE_INCLUDE", [stim.target_rec(-n_data)], 0)
    return c

In [ ]:
circuit = make_rep_circuit(p=0.01, rounds=3, distance=3)
print(circuit)

In [ ]:
circuit.diagram("timeline-svg")

In [ ]:
# Sanity check: no noise => all detectors silent, no observable flips.
noiseless = make_rep_circuit(p=0.0, rounds=5, distance=5)
det, obs = noiseless.compile_detector_sampler().sample(
    shots=5000, separate_observables=True
)
print(f"Detector firing rate at p=0:  {det.mean():.6f}")
print(f"Observable flip rate at p=0:  {obs.mean():.6f}")

In [ ]:
# Sanity check: no noise => all detectors silent, no observable flips.
noiseless = make_rep_circuit(p=0.01, rounds=5, distance=5)
det, obs = noiseless.compile_detector_sampler().sample(
    shots=5000, separate_observables=True
)
print(f"Detector firing rate:  {det.mean():.6f}")
print(f"Observable flip rate:  {obs.mean():.6f}")

<a id="sec-4"></a>

---

## Section 4: Simulation and decoding of the repetition code

We now decode the multi-round detector outcomes. As in Lab 7, we use **PyMatching**.

In [ ]:
def decode(circuit, shots):
    """Sample the circuit and decode with PyMatching.

    Returns the empirical logical error rate after correction and the
    uncorrected observable flip rate for reference.
    """
    matcher = pymatching.Matching.from_stim_circuit(circuit)
    sampler = circuit.compile_detector_sampler()
    det_events, obs_flips = sampler.sample(shots=shots, separate_observables=True)
    preds = matcher.decode_batch(det_events)
    obs_corrected = obs_flips ^ preds
    return obs_corrected.mean(), obs_flips.mean()

### Sweep 1 — physical error rate $p$, with $\mathrm{rounds} = d = 3$

Fixing the rounds equal to the code distance is the standard convention; we will see why in Section 5.

In [ ]:
plist = np.logspace(-3, -1, 11)
shots = 50_000

pL_qec_d3 = []
pL_noqec_d3 = []
for p in plist:
    c = make_rep_circuit(p=p, rounds=3, distance=3)
    pl, pl_no = decode(c, shots)
    pL_qec_d3.append(pl)
    pL_noqec_d3.append(pl_no)

plt.plot(plist, pL_qec_d3, "o-", label="with decoder")
plt.plot(plist, pL_noqec_d3, "^-", label="raw $Z_0$")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate $p_L$")
plt.title("Rep code (d=3, rounds=3) under circuit-level noise")
plt.legend()
plt.show()

### Sweep 2 — number of rounds, with $p$ fixed

For a memory experiment, the logical error rate per *shot* grows roughly linearly with the number of rounds (the longer we wait, the more chances something goes wrong). The slope is the **logical error rate per round**, the quantity hardware papers report.

In [ ]:
rounds_list = [1, 2, 3, 5, 7, 10, 15, 20]
p_fixed = 0.01
shots = 50_000

pL_rounds = []
for R in rounds_list:
    c = make_rep_circuit(p=p_fixed, rounds=R, distance=3)
    pl, _ = decode(c, shots)
    pL_rounds.append(pl)

plt.plot(rounds_list, pL_rounds, "o-")
plt.xlabel("Number of syndrome extraction rounds")
plt.ylabel(f"Logical error rate $p_L$ (per shot, $p={p_fixed}$)")
plt.title("Rep code logical error rate vs. memory time")
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 3 — Bit-flip code threshold across distances

**LER per round** $p_L^\mathrm{round}$:

Assume that a logical error occurs when an odd number of layers have errors.

$$
p_L = \frac{1 - (1 - 2p_L^\mathrm{round})^R}{2} ~\Longrightarrow~ p_L^\mathrm{round} = \frac{1 - (1 - 2p_L)^{1/R}}{2}
$$

In [ ]:
def get_ler_per_round(pL, R):
    return (1 - (1 - 2 * pL) ** (1 / R)) / 2

In [ ]:
plist = np.logspace(-2.5, -1, 9)
dlist = [3, 5, 7, 9]
shots = 30_000

pL_per_round = np.zeros((len(dlist), len(plist)))
for i, d in enumerate(dlist):
    for j, p in enumerate(plist):
        # TODO: build the bit-flip code circuit at this (p, d) with rounds = d
        # TODO: store the logical error rate per round in pL_per_round[i, j]
        # pL_per_round[i, j] = ...
        pass
    print(f"d = {d}: done")

# --- Plot ---
for i, d in enumerate(dlist):
    plt.plot(plist, pL_per_round[i], "o-", label=f"d = {d}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate $p_L$")
plt.legend()
plt.grid(True, which="both", alpha=0.2)
plt.show()

<a id="sec-5"></a>

---

## Section 5: Surface code threshold under circuit-level noise

We now move from the (one-dimensional) repetition code to the (two-dimensional) rotated surface code.

`stim.Circuit.generated("surface_code:rotated_memory_z", ...)` accepts four circuit-level noise parameters:

- `after_clifford_depolarization` — `DEPOLARIZE1`/`DEPOLARIZE2` after every Clifford gate.
- `after_reset_flip_probability` — `X_ERROR` after every reset.
- `before_measure_flip_probability` — `X_ERROR` before every measurement.
- **Important note:** stim implementation doesn't support idling noise natively.
    * One solution is to use `before_round_data_depolarization=N*p`, where `N` is the number of ticks in each round. 
    * Strictly, this is different from the original SD6 definition (more noisy), but let's ignore this here.

In [ ]:
circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    distance=5,
    rounds=5,
)
circuit.diagram("timeline-svg")

In [ ]:
def make_surface_circuit(p, d):
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=d,
        rounds=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=7 * p,  # A single round includes 7 ticks.
    )

In [ ]:
# Inspect a small example to see the same detector pattern as Section 3.
example = make_surface_circuit(p=0.005, d=3)
example.diagram("timeslice-svg")

### Threshold sweep

The classic surface-code threshold plot: for each distance $d$, the logical error rate is a curve as a function of $p$. The curves **cross** at the threshold $p_\mathrm{th}$ — below the crossing, larger codes do better; above it, larger codes do worse.

The cell below takes the longest in this lab. The bottleneck is the PyMatching call for $d = 9$, not the Stim sampler; reduce `shots` if you need a quicker run.

In [ ]:
plist = np.logspace(-3, -1.5, 9)
dlist = [3, 5, 7, 9]
shots = 10_000

pL_surface = np.zeros((len(dlist), len(plist)))
for i, d in enumerate(dlist):
    for j, p in enumerate(plist):
        syndrome_circuit = make_surface_circuit(p=p, d=d)
        pl, _ = decode(syndrome_circuit, shots)
        pL_surface[i, j] = get_ler_per_round(pl, d)  # LER per round
    print(f"d = {d}: done")

In [ ]:
for i, d in enumerate(dlist):
    plt.plot(plist, pL_surface[i], "o-", label=f"d = {d}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate per round $p_L$")
plt.title("Rotated surface code threshold (standard depolarizing model)")
plt.legend()
plt.grid(True, which="both", alpha=0.2)
plt.show()

The curves cross somewhere around $p \approx 1\%$ — this is the famous surface-code circuit-level threshold under the standard depolarizing model.